In [9]:
import tensorflow as tf
from tensorflow import keras


import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.metrics
from PIL import Image
import pandas as pd

# discontinued import tensorflow_addons as tfa
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import load_model
# from tensorflow.keras.regularizers import l2
# from tensorflow.keras.layers import Dropout, Dense, Flatten

from sklearn.utils import resample
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from collections import Counter


tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [2]:
base_dir = r"C:\Users\Aryan\Downloads\Dataset - train+val+test"
data = tf.keras.preprocessing.image_dataset_from_directory(base_dir)

Found 109309 files belonging to 3 classes.


In [3]:
print('Base directory --> ', os.listdir(base_dir))

Base directory -->  ['test', 'train', 'val']


In [4]:
# NOT EXCHANGED

train_dir = r"C:\Users\Aryan\Downloads\Dataset - train+val+test\train"
print("Train Directory --> ", os.listdir(train_dir))

validation_dir = r"C:\Users\Aryan\Downloads\Dataset - train+val+test\val"
print("Validation Directory --> ", os.listdir(validation_dir))

test_dir = r"C:\Users\Aryan\Downloads\Dataset - train+val+test\test"
print("Test Directory --> ", os.listdir(test_dir))

Train Directory -->  ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Validation Directory -->  ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Test Directory -->  ['CNV', 'DME', 'DRUSEN', 'NORMAL']


In [5]:
# Load image paths and labels
image_paths = []
labels = []

for class_label in ['CNV', 'drusen', 'DME', 'normal']:
    class_dir = os.path.join(train_dir, class_label)
    for img_name in os.listdir(class_dir):
        image_paths.append(os.path.join(class_dir, img_name))
        labels.append(class_label)

# Convert to numpy arrays
image_paths = np.array(image_paths)
labels = np.array(labels)

# Check class distribution
print(Counter(labels))



Counter({'normal': 35973, 'CNV': 26218, 'DME': 8118, 'drusen': 6206})


In [6]:

# Find minimum class size
min_class_size = min(Counter(labels).values())

# Function to undersample
def undersample(image_paths, labels, target_size):
    undersampled_paths = []
    undersampled_labels = []

    for class_label in np.unique(labels):
        # Get indices of the current class
        class_indices = np.where(labels == class_label)[0]
        
        # Downsample to target size
        undersampled_indices = resample(class_indices, 
                                        replace=False, 
                                        n_samples=target_size, 
                                        random_state=42)
        
        undersampled_paths.extend(image_paths[undersampled_indices])
        undersampled_labels.extend(labels[undersampled_indices])

    return np.array(undersampled_paths), np.array(undersampled_labels)

# Undersample to the size of the smallest class
image_paths_resampled, labels_resampled = undersample(image_paths, labels, min_class_size)

# Check new class distribution
print(Counter(labels_resampled))

Counter({'CNV': 6206, 'DME': 6206, 'drusen': 6206, 'normal': 6206})


In [7]:
from tensorflow.keras.applications import InceptionV3
inc = InceptionV3(input_shape=(224,224,3),weights='imagenet',include_top=False)
for i in inc.layers:
    i.trainable = False
print(inc.summary())

Model: "inception_v3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 111, 111, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 111, 111, 32  96         ['conv2d[0][0]']                 
 alization)                     )                                                      

In [32]:
model = tf.keras.models.Sequential([
    inc,
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(4, activation = 'softmax')
])

In [33]:
metrics = ['accuracy',
                tf.keras.metrics.AUC(),
               # tfa.metrics.CohenKappa(num_classes = 4),
               # tfa.metrics.F1Score(num_classes = 4),
                tf.keras.metrics.Precision(), 
                tf.keras.metrics.Recall()]
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = metrics)
print(model.summary())

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 inception_v3 (Functional)   (None, 5, 5, 2048)        21802784  
                                                                 
 flatten_4 (Flatten)         (None, 51200)             0         
                                                                 
 dense_7 (Dense)             (None, 4)                 204804    
                                                                 
Total params: 22,007,588
Trainable params: 204,804
Non-trainable params: 21,802,784
_________________________________________________________________
None


In [34]:
# train_datagen = ImageDataGenerator(rescale = 1./255)
# train_generator = train_datagen.flow_from_directory(
#     train_dir, target_size = (224,224), class_mode = 'categorical', 
#     batch_size = 500)

In [35]:
# # Function to load images
# def load_images(image_paths):
#     images = []
#     for path in image_paths:
#         img = load_img(path, target_size=(224, 224))  # Adjust target size as needed
#         img = img_to_array(img)
#         images.append(img)
#     return np.array(images)

# X_train = load_images(image_paths_resampled)
# y_train = labels_resampled

# # Normalize images
# X_train = X_train / 255.0

In [36]:
# Create a DataFrame from undersampled data
df = pd.DataFrame({'filename': image_paths_resampled, 'class': labels_resampled})

# Data Generator
datagen = ImageDataGenerator(rescale=1./255)  # Normalize pixel values

# Load images from file paths
train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),  # Adjust to match your model input size
    batch_size=64,
    class_mode="categorical",  # Use "binary" for 2 classes, "sparse" for integer labels
    shuffle=True
)

# Check class indices
print(train_generator.class_indices)


Found 24824 validated image filenames belonging to 4 classes.
{'CNV': 0, 'DME': 1, 'drusen': 2, 'normal': 3}


In [37]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)


In [38]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir, target_size = (224,224), class_mode = 'categorical', shuffle=False, 
    batch_size = 64)

Found 10933 images belonging to 4 classes.


In [39]:
validation_datagen = ImageDataGenerator(rescale = 1./255)
validation_generator = validation_datagen.flow_from_directory(
    validation_dir, target_size = (224,224), class_mode = 'categorical', 
    batch_size = 64)

Found 21861 images belonging to 4 classes.


In [40]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
# model.fit(train_generator, epochs=30, validation_data=validation_generator, )

In [43]:

model = load_model("RS_model_4_es.h5")  # Loads the entire model


In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch = (24824//64),
    epochs = 30,
    validation_data = validation_generator,
    validation_steps = (21861//64),
    #max_queue_size=100,
    #workers = 4 ,
    #verbose = 1
    callbacks=[early_stop])

Epoch 1/30
182/387 [=============>................] - ETA: 30s - loss: 0.8724 - accuracy: 0.8518 - auc: 0.9480 - precision: 0.8531 - recall: 0.8506  

In [ ]:
model.save('RS_model_5_es.h5')

In [ ]:
# if 'history' in locals():  # Check if history exists
#     import json
#     history_dict = history.history
#     with open('history.json', 'w') as f:
#         json.dump(history_dict, f)
#     print("History saved successfully.")
# else:
#     print("No history available to save.")

In [ ]:
# import json
# history_dict = history.history
# with open('history.json', 'w') as f:
#     json.dump(history_dict, f)

In [ ]:
print("metrics of Inception V3")
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.figure(figsize=(12,12))

plt.plot(epochs, acc, 'r', label = 'Training accuracy')
plt.plot(epochs, val_acc, 'b', label = 'Validation accuracy')
plt.title('Training & validation accuracy')
plt.legend()

plt.figure(figsize = (12,12))

plt.plot(epochs, loss, 'r', label = 'Training Loss')
plt.plot(epochs, val_loss, 'b', label = 'Validation Loss')
plt.title('Training $ validation loss')
plt.legend()


# Evaluating Test Data

In [ ]:
model.evaluate(test_generator)

# Classification Report

In [ ]:
test_steps_per_epoch = np.math.ceil(test_generator.samples / test_generator.batch_size)

predictions = model.predict(test_generator, steps = test_steps_per_epoch)

predicted_classes = np.argmax(predictions, axis=1)

In [ ]:
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

In [ ]:
report = sklearn.metrics.classification_report(true_classes, predicted_classes, target_names = class_labels)
print(report) 

# Confusion Matrix

In [ ]:
X_Label = ['CVN','DME','DRUSEN','NORMAL']
Y_Label = ['CVN','DME','DRUSEN','NORMAL']

cm = sklearn.metrics.confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(8,8))
sns.heatmap(cm, fmt='.0f', cmap="crest", annot=True, linewidths=0.2, xticklabels=X_Label, yticklabels=Y_Label)
plt.title('confusion matrix')
plt.xlabel('predicted value')
plt.ylabel('Truth value')
plt.show()
print(sklearn.metrics.confusion_matrix(true_classes, predicted_classes))